# 16 - LoRA fine-tuning (PEFT) - second post-hoc capacity control

**Why a second notebook.** NB15 fine-tuned the *embedder* contrastively (SetFit-style). This one does **parameter-efficient fine-tuning (LoRA)** of a small transformer end-to-end for classification - a different mechanism and the more marketable skill. If *both* independent approaches sharpen the concrete categories but leave the **triangle** flat, that is strong, convergent evidence the ceiling is the **labels, not the model**.

**What LoRA is.** The base transformer stays frozen; we train tiny low-rank adapter matrices injected into its attention layers (~1% of params). Same idea as full fine-tuning, far cheaper, less overfitting on small data.

**Protocol (same as NB15).** Train on `data/modelling/train.csv` (942), evaluate on `val.csv` (167). Nothing fitted on held-out. Same metrics, same triangle/concrete split, so numbers sit next to your 0.750 baseline and NB15.

**Triangle:** `policy_practice_research`, `political_environment_key_organisations`, `what_matters_ed`. **Concrete:** `edtech`, `four_nations`, `teacher_rrd`.

> Base model: `distilbert-base-uncased` (small, CPU-feasible). First run downloads ~270MB. Class imbalance handled with a weighted loss (matching the baseline's `class_weight='balanced'`).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sentence_transformers import SentenceTransformer
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

ROOT = Path.cwd() if (Path.cwd()/'data').exists() else Path.cwd().parent
DATA = ROOT/'data'/'modelling'
TEXT_COL, LABEL_COL = 'text_clean', 'target'
SEED = 42
BASE = 'distilbert-base-uncased'
TRIANGLE = ['policy_practice_research','political_environment_key_organisations','what_matters_ed']
CONCRETE = ['edtech','four_nations','teacher_rrd']

train = pd.read_csv(DATA/'train.csv'); val = pd.read_csv(DATA/'val.csv')
CLASSES = sorted(train[LABEL_COL].unique()); c2i = {c:i for i,c in enumerate(CLASSES)}
print('train', train.shape, '| val', val.shape, '|', len(CLASSES), 'classes')

In [ ]:
# --- metric helpers (identical to NB15) ---
def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    return dict(zip(CLASSES, f))

def summarise(name, y_true, y_pred, proba=None):
    pcf = per_class_f1(y_true, y_pred)
    out = {'model':name,
           'overall_macroF1':float(np.mean(list(pcf.values()))),
           'triangle_macroF1':float(np.mean([pcf[c] for c in TRIANGLE])),
           'concrete_macroF1':float(np.mean([pcf[c] for c in CONCRETE]))}
    if proba is not None:
        top2 = np.array(CLASSES)[np.argsort(proba, axis=1)[:, -2:]]
        out['top2_acc'] = float(np.mean([yt in row for yt,row in zip(y_true, top2)]))
    out.update({f'F1[{c}]': round(pcf[c],3) for c in CLASSES})
    return out

## 1. Baseline - frozen embedder + logistic-regression head
Same anchor as NB15, so all three (baseline / SetFit / LoRA) are comparable.

In [ ]:
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
Xtr = emb.encode(train[TEXT_COL].tolist(), show_progress_bar=True)
Xva = emb.encode(val[TEXT_COL].tolist(), show_progress_bar=True)
base_clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED).fit(Xtr, train[LABEL_COL])
base_proba = base_clf.predict_proba(Xva)
# align proba columns to CLASSES order
base_proba = base_proba[:, [list(base_clf.classes_).index(c) for c in CLASSES]]
base_row = summarise('frozen + logreg (baseline)', val[LABEL_COL].values, base_clf.predict(Xva), base_proba)
print({k:round(v,3) for k,v in base_row.items() if 'macroF1' in k or k=='top2_acc'})

## 2. LoRA fine-tune of distilbert
~10-20 min on CPU for 4 epochs. Lower `EPOCHS` to 1-2 for a quick first pass.

In [ ]:
from transformers import set_seed; set_seed(SEED)
EPOCHS = 4
tok = AutoTokenizer.from_pretrained(BASE)
def make_ds(df):
    d = Dataset.from_dict({'text': df[TEXT_COL].astype(str).tolist(),
                           'labels': [c2i[t] for t in df[LABEL_COL]]})
    return d.map(lambda b: tok(b['text'], truncation=True, padding='max_length', max_length=128), batched=True)
tr_ds, va_ds = make_ds(train), make_ds(val)

model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=len(CLASSES))
lcfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
                  target_modules=['q_lin','v_lin'])
model = get_peft_model(model, lcfg)
model.print_trainable_parameters()

# class weights to match the baseline's balanced setting
counts = train[LABEL_COL].value_counts().reindex(CLASSES).values.astype(float)
class_weights = torch.tensor((counts.sum()/(len(CLASSES)*counts)), dtype=torch.float32)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels,
                                                 weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(output_dir='/tmp/lora_out', num_train_epochs=EPOCHS,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=2e-4, report_to=[], logging_steps=50, seed=SEED)
trainer = WeightedTrainer(model=model, args=args, train_dataset=tr_ds)
trainer.train()
print('LoRA fine-tune done')

In [ ]:
pred = trainer.predict(va_ds)
logits = torch.tensor(pred.predictions)
proba = torch.softmax(logits, dim=1).numpy()
yhat = [CLASSES[i] for i in proba.argmax(1)]
lora_row = summarise('LoRA distilbert', val[LABEL_COL].values, yhat, proba)
print({k:round(v,3) for k,v in lora_row.items() if 'macroF1' in k or k=='top2_acc'})

## 3. Side by side (and bring in NB15 if you have it)

In [ ]:
rows = [base_row, lora_row]
# optionally pull NB15's SetFit result if it was saved
setfit_csv = ROOT/'outputs'/'setfit_vs_baseline_val.csv'
if setfit_csv.exists():
    sf = pd.read_csv(setfit_csv)
    sf_ft = sf[sf['model'].str.contains('fine-tune')]
    if len(sf_ft):
        rows.append({**sf_ft.iloc[0].to_dict()})
cmp = pd.DataFrame(rows).set_index('model')
cols = [c for c in ['overall_macroF1','triangle_macroF1','concrete_macroF1','top2_acc'] if c in cmp.columns]
print(cmp[cols].round(3).to_string())
print(f"\nTriangle change from LoRA: {lora_row['triangle_macroF1']-base_row['triangle_macroF1']:+.3f}")
(ROOT/'outputs').mkdir(exist_ok=True)
cmp.round(4).to_csv(ROOT/'outputs'/'lora_vs_baseline_val.csv')
print('saved -> outputs/lora_vs_baseline_val.csv')

## 4. Your read first
- Did LoRA move **overall** macro F1 vs the 0.750 baseline?
- **Concrete** trio up (expected)?
- **Triangle** trio - flat / down / up? Flat is the thesis: a different fine-tuning mechanism *also* can't separate categories the data does not separate.
- Compare to NB15: do the two independent methods **agree** on the triangle? Convergent no-gain is the strong result.

**Caveats:** small data + a 270MB general-English distilbert (not domain-adapted), so do not over-read a single run; seed variance applies. The honest claim is about the *pattern* (concrete sharpens, triangle does not), echoed across SetFit, LoRA and the earlier full DistilBERT fine-tune (0.750, no gain).

_If a Trainer cell errors on transformers 5.x, paste the message - likely a one-line arg rename._